In [2]:
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [6]:
r = requests.get('https://httpbin.org/user-agent')
useragent = r.json()['user-agent']
headers = {'User-Agent': useragent,
           'From': 'jbm2rt@virginia.edu'}

In [7]:
url = 'https://www.rottentomatoes.com/browse/movies_in_theaters/sort:a_z?page=5'
r = requests.get(url, headers=headers)
r

<Response [200]>

In [9]:
mysoup = BeautifulSoup(r.text, 'html.parser')

In [12]:
movielist = mysoup.find_all('a', attrs={'data-track':'scores'})

In [15]:
len(movielist)
'https://rottentomatoes.com' + movielist[3]['href']

'https://rottentomatoes.com/m/all_we_imagine_as_light'

In [17]:
links = ['https://rottentomatoes.com' + m['href'] for m in movielist]
links

['https://rottentomatoes.com/m/2121',
 'https://rottentomatoes.com/m/a_complete_unknown',
 'https://rottentomatoes.com/m/a_real_pain',
 'https://rottentomatoes.com/m/all_we_imagine_as_light',
 'https://rottentomatoes.com/m/anora',
 'https://rottentomatoes.com/m/armand',
 'https://rottentomatoes.com/m/attack_on_titan_the_last_attack',
 'https://rottentomatoes.com/m/babygirl_2024',
 'https://rottentomatoes.com/m/becoming_led_zeppelin',
 'https://rottentomatoes.com/m/blades_in_the_darkness',
 'https://rottentomatoes.com/m/bring_them_down',
 'https://rottentomatoes.com/m/captain_america_brave_new_world',
 'https://rottentomatoes.com/m/cleaner_2025',
 'https://rottentomatoes.com/m/companion_2025',
 'https://rottentomatoes.com/m/conclave',
 'https://rottentomatoes.com/m/creation_of_the_gods_ii_demon_force',
 'https://rottentomatoes.com/m/dark_nuns',
 'https://rottentomatoes.com/m/den_of_thieves_2_pantera',
 'https://rottentomatoes.com/m/ernest_cole_lost_and_found',
 'https://rottentomatoes.c

In [18]:
url = 'https://rottentomatoes.com/m/a_complete_unknown'
r = requests.get(url, headers=headers)
mysoup = BeautifulSoup(r.text, 'html.parser')

In [20]:
titletag = mysoup.find('h1', attrs={'class':'unset'})
title = titletag.text.strip()
mydict = {'title': title}
mydict

{'title': 'A Complete Unknown'}

In [21]:
descriptag = mysoup.find('rt-text', attrs={'slot':'content'})
description = descriptag.text.strip()
mydict['description'] = description
mydict

{'title': 'A Complete Unknown',
 'description': "New York, 1961. Against the backdrop of a vibrant music scene and tumultuous cultural upheaval, an enigmatic 19-year-old from Minnesota arrives with his guitar and revolutionary talent, destined to change the course of American music. He forges intimate relationships with music icons of Greenwich Village on his meteoric rise, culminating in a groundbreaking and controversial performance that reverberates worldwide. Timothée Chalamet stars and sings as Bob Dylan in James Mangold's A COMPLETE UNKNOWN, the electric true story behind the rise of one of the most iconic singer-songwriters in history."}

In [22]:
def scrape_one(url):
    mydict = {}
    r = requests.get(url, headers=headers)
    mysoup = BeautifulSoup(r.text, 'html.parser')
 
    mydict['Title'] = mysoup.find('h1', 'unset').text.strip()
 
    sum = mysoup.find('meta', attrs={'name':'description'})
    mydict['Synopsis'] = sum['content']
 
    consensus = mysoup.find('div', 'consensus')
    try:
        mydict['Critic Consensus'] = consensus.p.text
    except:
        mydict['Critic Consensus'] = ''
 
    mydict['Critics Score'] = mysoup.find('rt-text', slot="criticsScore").text.strip()
    mydict['Audience Score'] = mysoup.find('rt-text', slot="audienceScore").text.strip()
 
    cat = mysoup.find_all('div',"category-wrap")
    cats = [x.text.strip().split('\n\n\n') for x in cat]
    for c in cats:
        mydict[c[0]] = c[1].replace('\n','')
 
    return mydict

In [23]:
scrape_one(links[3])

{'Title': 'All We Imagine as Light',
 'Synopsis': "The light, the lives, and the textures of contemporary, working-class Mumbai are explored and celebrated by writer/director Payal Kapadia, who won the Grand Prize at this year's Cannes Film Festival for her revelatory fiction feature debut. Centering on two roommates who also work together in a city hospital--head nurse Prabha (Kani Kusruti) and recent hire Anu (Divya Prabha)--plus their coworker, cook Parvaty (Chhaya Kadam), Kapadia's film alights on moments of connection and heartache, hope and disappointment. Prabha, her husband from an arranged marriage living in faraway Germany, is courted by a doctor at her hospital; Anu carries on a romance with a Muslim man, which she must keep a secret from her strict Hindu family; Parvaty finds herself dealing with a sudden eviction from her apartment. Kapadia captures the bustle of the metropolis and the open-air tranquility of a seaside village with equal radiance, articulated by her superb

In [24]:
full_movie_list = [scrape_one(url) for url in links]

In [28]:
pd.DataFrame.from_records(full_movie_list)

,Title,Synopsis,Critic Consensus,Critics Score,Audience Score,Director,Producer,Screenwriter,Distributor,Genre,Original Language,Release Date (Theaters),Box Office (Gross USA),Runtime,Production Co,Rating,Sound Mix,Aspect Ratio,Release Date (Streaming),Rerelease Date (Theaters)
0,2121,100 years from now when humans are forced unde...,,,,Serpil Altin,"Korhan Ugur, Serpil Altin","Korhan Ugur, Serpil Altin",Indican Pictures,"Mystery & Thriller, Drama, Sci-Fi",English,"Jan 31, 2025, Limited",$7.9K,1h 32m,NaN,NaN,NaN,NaN,NaN,NaN
1,A Complete Unknown,"New York, 1961. Against the backdrop of a vibr...",Charged by Timothée Chalamet's electric perfor...,81%,96%,James Mangold,"Fred Berger, Bob Bookman, Timothée Chalamet, A...","James Mangold, Jay Cocks",Searchlight Pictures,"Biography, Drama, Music",English,"Dec 25, 2024, Wide",$69.0M,2h 21m,"Searchlight Pictures, The Picture Company, Ver...",R (Language),"Dolby Atmos, Dolby Digital",Digital 2.39:1,NaN,NaN
2,A Real Pain,Mismatched cousins David (Jesse Eisenberg) and...,Led by a scene-stealing turn from Kieran Culki...,96%,81%,Jesse Eisenberg,"Jesse Eisenberg, Ali Herting, Dave McCary, Ewa...",Jesse Eisenberg,Searchlight Pictures,"Comedy, Drama",English,"Nov 15, 2024, Wide",$8.3M,1h 29m,"Topic Studios, Fruit Tree, Extreme Emotions",R (Some Drug Use|Language Throughout),NaN,Flat (1.85:1),"Dec 31, 2024",NaN
3,All We Imagine as Light,"The light, the lives, and the textures of cont...",Capturing the here and now of modern India wit...,100%,68%,Payal Kapadia,NaN,Payal Kapadia,Sideshow / Janus Films,Drama,Malayalam,"Nov 15, 2024, Limited",$1.0M,1h 58m,"Chalk and Cheese, Another Birth, arte France C...",NaN,Dolby Digital,Flat (1.66:1),"Feb 4, 2025",NaN
4,Anora,Sean Baker's Palme d'Or winner ANORA is an aud...,Another marvelous chronicle of America's striv...,94%,90%,Sean Baker,"Alex Coco, Samantha Quan, Sean Baker",Sean Baker,NEON,"Comedy, Drama, Romance",English,"Nov 8, 2024, Wide",$15.3M,2h 19m,Cre Film,R (Graphic Nudity|Drug Use|Pervasive Language|...,Dolby Digital,Digital 2.39:1,"Dec 17, 2024",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,Universal Language,In a mysterious and surreal interzone somewher...,Bridging two nations with a clever and surreal...,97%,,Matthew Rankin,Sylvain Corbeil,"Pirouz Nemati, Ila Firouzabadi, Matthew Rankin...",Oscilloscope Laboratories,Comedy,Persian,"Feb 14, 2025, Limited",NaN,1h 29m,Metafilms,NaN,NaN,Flat (1.85:1),NaN,NaN
67,Vermiglio,"The lush and breathtaking beauty of the Alps, ...","Painterly and patient, Vermiglio carefully obs...",94%,,Maura Delpero,"Carole Baraton, Tatjana Kozar, Francesca Andreoli",Maura Delpero,Sideshow / Janus Films,Drama,Italian,"Dec 25, 2024, Limited",$158.2K,1h 59m,"RAI Cinema, Cinedora, Versus Production, Charades",NaN,NaN,NaN,NaN,NaN
68,Wicked,"Wicked, the untold story of the witches of Oz,...",Defying gravity with its magical pairing of Cy...,88%,95%,Jon M. Chu,"Marc Platt, David Stone","Winnie Holzman, Dana Fox",Universal Pictures,"Kids & Family, Musical, Fantasy, Adventure",English,"Nov 22, 2024, Wide",$471.9M,2h 40m,Marc Platt Productions,PG (Some Scary Action|Brief Suggestive Materia...,Dolby Atmos,Digital 2.39:1,"Dec 31, 2024",NaN
69,Wolf Man,From Blumhouse and visionary writer-director L...,Director Leigh Whannell's attempt at bringing ...,50%,56%,Leigh Whannell,"Jason Blum, Ryan Gosling","Leigh Whannell, Corbett Tuck, Lauren Schuker B...",Universal Pictures,"Horror, Mystery & Thriller",English,"Jan 17, 2025, Wide",$20.6M,1h 43m,"Blumhouse Productions, Universal Pictures, Way...",R (Grisly Images|Bloody Violent Content|Some L...,Dolby Atmos,Digital 2.39:1,"Feb 4, 2025",NaN
